# Génération de données simulées


Objectif : génération reproductible et paramétrable des données simulées avec colonnes conformes (`annee, semestre, anonymat, ue, credit, nom_prenoms, sexe, note`).

In [1]:
from __future__ import annotations

import random
import hashlib
from pathlib import Path
from typing import Dict, List, Optional

import numpy as np
import pandas as pd

import seaborn as sns
import matplotlib.pyplot as plt

#  Reproductibilité 
RNG_SEED = 42
rng = np.random.default_rng(RNG_SEED)

sns.set_theme(style="whitegrid")

## Paramètres globaux

In [2]:
#  Cohortes 
COHORTES: List[int] = [2018, 2019, 2020, 2021, 2022, 2023]
ETUDIANTS_PAR_COHORTE: Dict[int, int] = {
    2018: 80,
    2019: 95,
    2020: 100,
    2021: 150,
    2022: 135,
    2023: 120,
}

#  Paramètres pédagogiques ─
SEUIL_REUSSITE     = 10
SEMESTRES_PAR_CYCLE = 10

CREDITS_PAR_SEMESTRE: Dict[int, int] = {
    1: 31, 2: 29,
    3: 32, 4: 30,
    5: 27, 6: 31,
    7: 30, 8: 31,
    9: 33, 10: 26,
}


CREDITS_POSSIBLES         = [2, 3, 4]
NB_UES_CIBLES_PAR_SEM = 11   

PREFIXES_UE = ["MTH", "PHY", "INF", "CHE", "ECO", "HIS", "GEO", "ANG", "STT", "BIO", "ART"]

DATA_DIR = Path("../data").resolve()
DATA_DIR.mkdir(parents=True, exist_ok=True)

#print(f"3 premières années : {sum(v for k,v in CREDITS_PAR_SEMESTRE.items() if k<=6)} crédits")
#print(f"2 dernières années : {sum(v for k,v in CREDITS_PAR_SEMESTRE.items() if k>6)} crédits")
#print(f"Total              : {sum(CREDITS_PAR_SEMESTRE.values())} crédits")

## Noms & prénoms

In [3]:
NOMS = [
    "Diallo", "Traoré", "Koné", "Diabaté", "Coulibaly", "Keita", "Ouattara", "Barry",
    "Camara", "Sow", "Bâ", "Bamba", "Berg", "Berthé", "Bissikolo", "Bogolan", "Boly",
    "Bouna", "Cissé", "Dabo", "Diarra", "Djité", "Fofana", "Gakou", "Gano", "Gassama",
    "Guindo", "Haidara", "Kanté", "Kassé", "Kéita", "Koman", "Konaté", "Kouriba",
    "Kouyaté", "Maïga", "Makan", "Mamadou", "Sacko", "Sangaré", "Sidibé", "Sissoko",
    "Sylla", "Tandja", "Tao", "Touré", "Wagaté", "Yattabaye", "Zongo", "Zouma",
    "Zoumana", "Zouolé", "Alassane", "Alou", "Amadou", "Bakary", "Bintou", "Cheick",
    "Djibril", "Fanta", "Habib", "Ibrahim", "Issa", "Karim", "Lassana", "Mahamadou",
    "Moussa", "Oumar", "Salif", "Souleymane", "Yacouba", "Younoussa", "Zacharie",
    "Abdou", "Abdoulaye", "Adama", "Aïcha", "Aminata", "Awa", "Fatoumata", "Hadja",
    "Halima", "Hawa", "Kadiatou", "Lala", "Mariam", "Mata", "Nabintou", "Nafissa",
    "Nana", "Oumou", "Rabi", "Rahmatou", "Sali", "Salimata", "Sata", "Sitan",
]

PRENOMS_M = [
    "Moussa", "Ibrahim", "Abdoulaye", "Mohamed", "Oumar", "Yacouba", "Seydou", "Adama",
    "Bakary", "Cheick", "Djibril", "Fodé", "Habib", "Issa", "Karim", "Lassana",
    "Mahamadou", "Mamadou", "Salif", "Souleymane", "Younoussa", "Zacharie", "Abdou",
    "Lamine", "Modibo", "Ousmane", "Samba", "Sidi", "Tiécoura", "Vieux", "Youssouf",
    "Alhassane", "Amadou", "Brahima", "Boureima", "Daouda", "Elhadj", "Fousseyni",
    "Harouna", "Issiaka", "Kassim", "Ladji", "Mahamane", "Mamane", "Nouhou",
    "Rachid", "Sada", "Soumaïla", "Tidjani", "Zakari",
]

PRENOMS_F = [
    "Awa", "Aminata", "Fatou", "Mariam", "Salimata", "Kadidia", "Kadiatou", "Aïcha",
    "Bintou", "Fanta", "Fatoumata", "Hadja", "Halima", "Hawa", "Nabintou", "Nafissa",
    "Oumou", "Rabi", "Rahmatou", "Sali", "Sata", "Sitan", "Ahou", "Akissi", "Alima",
    "Kadi", "Koudia", "Mawa", "M'mah", "Nafissatou", "Nana", "Yasmine", "Zana",
    "Zara", "Zénab", "Zénabou", "Zou", "Zouley", "Adama", "Aïssatou", "Djénéba",
    "Halimatou", "Aïda", "Aïssata", "Assa", "Assétou", "Djénébou",
]

## Catalogue des UEs

In [4]:
def split_credits(
    rng: np.random.Generator,
    total: int,
    pool: List[int],
    n: int,
) -> Optional[List[int]]:
    mn, mx = min(pool), max(pool)
    if not (n * mn <= total <= n * mx):
        return None

    vals = [mn] * n
    remainder = total - n * mn
    extras = sorted({c - mn for c in pool if c > mn}, reverse=True)

    for idx in rng.permutation(n):
        for extra in extras:
            if extra <= remainder:
                vals[idx] += extra
                remainder -= extra
                break
        if remainder == 0:
            break

    return vals if remainder == 0 else None


def generer_catalogue_ue(
    rng: np.random.Generator,
    credits_par_semestre: Dict[int, int] = CREDITS_PAR_SEMESTRE,
    CREDITS_POSSIBLES: List[int] = CREDITS_POSSIBLES,
    n_ues_cible: int = NB_UES_CIBLES_PAR_SEM,
) -> pd.DataFrame:
    """
    Génère un catalogue d'UEs 
    """
    codes_set: set = set()
    lignes = []

    for sem, credit_total in credits_par_semestre.items():
        annee_idx    = (sem - 1) // 2 + 1
        sem_in_annee = 1 if sem % 2 == 1 else 2

        # Trouver n_ues autour de la cible tel que la partition existe
        credits_sem: Optional[List[int]] = None
        n_ues = n_ues_cible
        for delta in [0, 1, -1, 2, -2, 3, -3, 4, 5]:
            candidate = n_ues_cible + delta
            if candidate < 1:
                continue
            result = split_credits(rng, credit_total, CREDITS_POSSIBLES, candidate)
            if result is not None:
                n_ues = candidate
                credits_sem = result
                break

        if credits_sem is None:  
            n_ues = credit_total // 3
            credits_sem = [3] * n_ues
            credits_sem[0] += credit_total - sum(credits_sem)

        rng.shuffle(credits_sem)

        for seq, cr in enumerate(credits_sem):
            for _ in range(50):
                prefix = rng.choice(PREFIXES_UE)
                code = f"{prefix}{annee_idx}{sem_in_annee}{seq:02d}"
                if code not in codes_set:
                    codes_set.add(code)
                    break
            else:
                code = f"{rng.choice(PREFIXES_UE)}{annee_idx}{sem_in_annee}{seq:03d}"
                codes_set.add(code)

            lignes.append({
                "ue":            code,
                "semestre":      sem,
                "credit":        cr,
                "moyenne_cible": round(float(rng.uniform(8, 14)), 2),
                "ecart_cible":   round(float(rng.uniform(2, 4)),  2),
            })

    return pd.DataFrame(lignes)


catalogue_ue = generer_catalogue_ue(rng)

#  Vérification des totaux
#tot_3 = catalogue_ue[catalogue_ue["semestre"] <= 6]["credit"].sum()
#tot_2 = catalogue_ue[catalogue_ue["semestre"] >  6]["credit"].sum()
#print(f"Total crédits 3 premières années : {tot_3}")
#print(f"Total crédits 2 dernières années : {tot_2}")
#print(f"Total général                    : {tot_3 + tot_2}")
#print(f"Nombre d'UEs                     : {len(catalogue_ue)}")

In [5]:
# Détail par semestre
catalogue_ue.groupby("semestre").agg(
    n_ues   = ("ue",     "count"),
    credits = ("credit", "sum"),
)

,n_ues,credits
semestre,,
1,11,31
2,11,29
3,11,32
4,11,30
5,11,27
6,11,31
7,11,30
8,11,31
9,11,33


In [6]:
catalogue_ue.head()

,ue,semestre,credit,moyenne_cible,ecart_cible
0,ANG1100,1,2,12.94,2.89
1,ECO1101,1,2,11.33,2.13
2,INF1102,1,2,12.97,3.26
3,PHY1103,1,4,10.13,3.94
4,STT1104,1,2,13.36,3.56


## Génération des étudiants

In [7]:
def map_cohorte_letter(cohorte: int, cohortes_list: List[int]) -> str:
    idx = sorted(set(cohortes_list)).index(cohorte)
    return chr(65 + idx % 26)


def hash_anonymat(carte: str, salt: str = "anonymat1") -> str:
    h = hashlib.sha256(f"{carte}-{salt}".encode()).hexdigest()
    return "A" + h[:7].upper()


def generer_etudiants(
    rng: np.random.Generator,
    n: int,
    cohorte: int,
    cohortes_list: List[int],
    salt: str = "anonymat1",
) -> pd.DataFrame:
    sexes    = rng.choice(["M", "F"], size=n, p=[0.8, 0.2])
    prefix   = map_cohorte_letter(cohorte, cohortes_list)
    cartes   = [f"{prefix}{i:05d}" for i in range(1, n + 1)]
    noms     = rng.choice(NOMS, size=n)
    prenoms  = [rng.choice(PRENOMS_M if s == "M" else PRENOMS_F) for s in sexes]
    niveau   = rng.normal(loc=0, scale=2, size=n)
    anonymats = [hash_anonymat(c, salt) for c in cartes]

    df = pd.DataFrame({
        "carte":       cartes,
        "anonymat":    anonymats,
        "nom_prenoms": [f"{n} {p}" for n, p in zip(noms, prenoms)],
        "sexe":        sexes,
        "niveau":      niveau,
    })

    if df["carte"].nunique() != n:
        raise ValueError("Doublons détectés dans les numéros de carte")
    if df["anonymat"].nunique() != n:
        raise ValueError("Doublons détectés dans les anonymats (vérifier le sel)")

    return df

## Génération des notes (vectorisée)


In [8]:
def generer_donnees_cohorte(
    rng: np.random.Generator,
    annee_entree: int,
    etudiants: pd.DataFrame,
    catalogue_ue: pd.DataFrame,
) -> pd.DataFrame:
    """Génère toutes les lignes notes×étudiants×UEs pour une cohorte (vectorisé)."""
    all_lignes: List[dict] = []
    niveaux = etudiants["niveau"].to_numpy()

    for i in range(5):  # 5 années académiques = 10 semestres
        annee_acad = f"{annee_entree + i}-{annee_entree + i + 1}"
        for sem in (2 * i + 1, 2 * i + 2):
            ue_sem = catalogue_ue[catalogue_ue["semestre"] == sem]
            for ue_ligne in ue_sem.itertuples(index=False):
                notes = np.clip(
                    rng.normal(
                        loc=ue_ligne.moyenne_cible + niveaux,
                        scale=ue_ligne.ecart_cible,
                    ),
                    0.0, 20.0,
                ).round(2)

                for j, etu in enumerate(etudiants.itertuples(index=False)):
                    all_lignes.append({
                        "annee":       annee_acad,
                        "semestre":    sem,
                        "anonymat":    etu.anonymat,
                        "carte":       etu.carte,
                        "ue":          ue_ligne.ue,
                        "credit":      int(ue_ligne.credit),
                        "nom_prenoms": etu.nom_prenoms,
                        "sexe":        etu.sexe,
                        "note":        float(notes[j]),
                    })

    return pd.DataFrame(all_lignes)


def generer_donnees_completes(
    rng: np.random.Generator,
    cohortes: List[int],
    effectifs: Dict[int, int],
    catalogue_ue: pd.DataFrame,
) -> pd.DataFrame:
    frames = []
    for annee_entree in cohortes:
        etudiants = generer_etudiants(
            rng, effectifs.get(annee_entree, 120),
            cohorte=annee_entree, cohortes_list=cohortes,
        )
        df_c = generer_donnees_cohorte(rng, annee_entree, etudiants, catalogue_ue)
        df_c["cohorte"] = annee_entree
        frames.append(df_c)
        print(f"  Cohorte {annee_entree} ✓  ({len(df_c):,} lignes)")
    return pd.concat(frames, ignore_index=True)

In [9]:
df = generer_donnees_completes(rng, COHORTES, ETUDIANTS_PAR_COHORTE, catalogue_ue)
df.head()

  Cohorte 2018 ✓  (8,800 lignes)
  Cohorte 2019 ✓  (10,450 lignes)
  Cohorte 2020 ✓  (11,000 lignes)
  Cohorte 2021 ✓  (16,500 lignes)
  Cohorte 2022 ✓  (14,850 lignes)
  Cohorte 2023 ✓  (13,200 lignes)


,annee,semestre,anonymat,carte,ue,credit,nom_prenoms,sexe,note,cohorte
0,2018-2019,1,A5646F46,A00001,ANG1100,2,Adama Kadidia,F,13.79,2018
1,2018-2019,1,A2BDCDD8,A00002,ANG1100,2,Kanté Ousmane,M,11.08,2018
2,2018-2019,1,A6BAFDCB,A00003,ANG1100,2,Sidibé Rabi,F,12.16,2018
3,2018-2019,1,A7AC0A37,A00004,ANG1100,2,Wagaté Ousmane,M,14.86,2018
4,2018-2019,1,AD6B6E40,A00005,ANG1100,2,Touré Assétou,F,11.35,2018


## Sanity checks

In [10]:
summary = {
    "lignes":           len(df),
    "ues_uniques":      df["ue"].nunique(),
    "semestres":        sorted(df["semestre"].unique()),
    "cohortes":         sorted(df["cohorte"].unique()),
    "annees_couvertes": df["annee"].nunique(),
    "min_note":         df["note"].min(),
    "max_note":         df["note"].max(),
}
summary

{'lignes': 74800,
 'ues_uniques': 110,
 'semestres': [np.int64(1),
  np.int64(2),
  np.int64(3),
  np.int64(4),
  np.int64(5),
  np.int64(6),
  np.int64(7),
  np.int64(8),
  np.int64(9),
  np.int64(10)],
 'cohortes': [np.int64(2018),
  np.int64(2019),
  np.int64(2020),
  np.int64(2021),
  np.int64(2022),
  np.int64(2023)],
 'annees_couvertes': 10,
 'min_note': np.float64(0.0),
 'max_note': np.float64(20.0)}

In [11]:
# Vérification crédits par semestre (doivent correspondre exactement)
check = (
    catalogue_ue
    .groupby("semestre")["credit"]
    .sum()
    .rename("credits_obtenus")
    .to_frame()
)
check["credits_attendus"] = pd.Series(CREDITS_PAR_SEMESTRE)
check["ok"] = check["credits_obtenus"] == check["credits_attendus"]
assert check["ok"].all(), "Incohérence dans les crédits !"
check

,credits_obtenus,credits_attendus,ok
semestre,,,
1,31,31,True
2,29,29,True
3,32,32,True
4,30,30,True
5,27,27,True
6,31,31,True
7,30,30,True
8,31,31,True
9,33,33,True


## Sauvegarde CSV + Parquet

In [12]:
csv_path     = DATA_DIR / "donnees_generees.csv"
parquet_path = DATA_DIR / "donnees_generees.parquet"

df.to_csv(csv_path, index=False)
df.to_parquet(parquet_path, index=False)

csv_path, parquet_path

(PosixPath('/home/liv3st/Desktop/ProjAnalyseS6/data/donnees_generees.csv'),
 PosixPath('/home/liv3st/Desktop/ProjAnalyseS6/data/donnees_generees.parquet'))